# Shadow Price Prediction - Main Pipeline

This notebook serves as the main entry point for the Shadow Price Prediction pipeline. 
It demonstrates how to configure and run the pipeline using a **Stacking Ensemble** approach.

In [1]:
from pbase.config.ray import init_ray
import shadow_price_prediction

# Initialize Ray for parallel processing
extra_modules = [shadow_price_prediction]
init_ray(address='ray://10.8.0.36:10001', extra_modules=extra_modules)

import pandas as pd
from shadow_price_prediction import (
    ShadowPricePipeline,
    PredictionConfig,
    FeatureConfig,
    ModelConfig,
    ModelSpec,
    EnsembleConfig,
    TrainingConfig,
    ThresholdConfig,
    AnomalyDetectionConfig,
    DataPathConfig,
    # Models
    XGBClassifier, XGBRegressor,
    LogisticRegression, ElasticNet,
    StackingModel
)

2025-11-19 10:32:13,134	INFO client_builder.py:242 -- Passing the following kwargs to ray.init() on the server: log_to_driver
2025-11-19 10:32:13,533	INFO packaging.py:588 -- Creating a file package for local module '/home/xyz/workspace/pbase/src/pbase'.
2025-11-19 10:32:13,652	INFO packaging.py:380 -- Pushing file package 'gcs://_ray_pkg_99f88ab2ee9d98c2.zip' (8.64MiB) to Ray cluster...
2025-11-19 10:32:13,720	INFO packaging.py:393 -- Successfully pushed file package 'gcs://_ray_pkg_99f88ab2ee9d98c2.zip'.
2025-11-19 10:32:13,728	INFO packaging.py:588 -- Creating a file package for local module '/home/xyz/workspace/research-spice-shadow-price-pred/src/shadow_price_prediction'.
2025-11-19 10:32:13,730	INFO packaging.py:380 -- Pushing file package 'gcs://_ray_pkg_e20f5e2076e433fc.zip' (0.21MiB) to Ray cluster...
2025-11-19 10:32:13,733	INFO packaging.py:393 -- Successfully pushed file package 'gcs://_ray_pkg_e20f5e2076e433fc.zip'.
SIGTERM handler is not set because current thread is not 

## 1. Configuration

Define the scope, data paths, and feature set.

In [2]:
# 1.1 Date and Market Scope
period_type = 'f0'
class_type = 'onpeak'
market_round = 1

# 1.2 Data Paths
density_path_template = (
    '/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/density/'
    'auction_month={auction_month}/market_month={market_month}/'
    'market_round={market_round}/outage_date={outage_date}'
)

constraint_path_template = (
    '/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/constraint_info/'
    'auction_month={auction_month}/market_round={market_round}/'
    'period_type={period_type}/class_type={class_type}'
)

# 1.3 Feature Configuration
common_features = [
    # Raw Density Shape
    '110', '105', '100', '95', '90',
    # Probability Mass Intervals (Diffs)
    '105_diff', '100_diff', '95_diff', '90_diff',
    '85_diff', '80_diff', '70_diff', '60_diff',
    # Engineered Risk Features
    'prob_overload', 'risk_ratio', 'curvature_100',
    'prob_exceed_110', 'prob_exceed_105', 'prob_exceed_100', 'prob_exceed_95', 'prob_exceed_90',
    'log_density_100',
    # Interaction Features
    'interaction_risk_overload', 'interaction_curvature_exceed'
]

feature_config = FeatureConfig(
    step1_features=common_features,
    step2_features=common_features
)

## 2. Model Configuration (Stacking Ensemble)

Here we define a **Stacking Ensemble**.

**How it works within `EnsembleConfig`:**
The `EnsembleConfig` expects a list of `ModelSpec` objects. To use Stacking, we simply create a `ModelSpec` where the `model_class` is `StackingModel`. 

If we want the Stacking Model to be the *only* model used, we give it a weight of `1.0` and provide no other models in the list.

In [ ]:
# Toggle between Stacking and Standard Ensemble
USE_STACKING = False  # Set to True for Stacking, False for Standard Ensemble

if USE_STACKING:
    # ==========================================
    # OPTION 1: Stacking Ensemble
    # ==========================================
    print("Using Stacking Ensemble...")
    
    # Define Base Models for Stacking
    xgb_base_clf = XGBClassifier(
        n_estimators=200, 
        max_depth=4, 
        learning_rate=0.1, 
        n_jobs=1, 
        eval_metric='logloss'
    )
    
    lr_base_clf = LogisticRegression(
        max_iter=1000, 
        class_weight='balanced', 
        n_jobs=1
    )
    
    # Define Stacking Classifier Spec
    stacking_clf_spec = ModelSpec(
        model_class=StackingModel,
        config=ModelConfig(params={
            'base_models': [xgb_base_clf, lr_base_clf],
            'meta_model': LogisticRegression(),
            'use_proba': True,
            'split_by_groups': True,  # Split CV folds by auction_month
        }),
        weight=1.0
    )
    
    # Define Base Models for Regression Stacking
    xgb_base_reg = XGBRegressor(
        n_estimators=200, 
        max_depth=4, 
        learning_rate=0.1, 
        n_jobs=1
    )
    
    enet_base_reg = ElasticNet(
        alpha=1.0, 
        l1_ratio=0.5
    )
    
    # Define Stacking Regressor Spec
    stacking_reg_spec = ModelSpec(
        model_class=StackingModel,
        config=ModelConfig(params={
            'base_models': [xgb_base_reg, enet_base_reg],
            'meta_model': XGBRegressor(n_estimators=50, max_depth=3),
            'use_proba': False,
            'split_by_groups': True,  # Split CV folds by auction_month
        }),
        weight=1.0
    )
    
    ensemble_config = EnsembleConfig(
        default_classifiers=[stacking_clf_spec],
        branch_classifiers=[stacking_clf_spec], 
        default_regressors=[stacking_reg_spec],
        branch_regressors=[stacking_reg_spec]
    )

else:
    # ==========================================
    # OPTION 2: Standard Ensemble (XGBoost + Linear)
    # ==========================================
    print("Using Standard Ensemble (XGBoost + Linear)...")
    
    # Standard Classifier Ensemble
    xgb_clf_spec = ModelSpec(
        model_class=XGBClassifier,
        config=ModelConfig(params={
            'n_estimators': 200,
            'max_depth': 4,
            'learning_rate': 0.1,
            'n_jobs': 1,
            'eval_metric': 'logloss'
        }),
        weight=0.5
    )
    
    lr_clf_spec = ModelSpec(
        model_class=LogisticRegression,
        config=ModelConfig(params={
            'C': 1.0,
            'max_iter': 1000,
            'class_weight': 'balanced',
            'n_jobs': 1
        }),
        weight=0.5
    )
    
    # Standard Regressor Ensemble
    xgb_reg_spec = ModelSpec(
        model_class=XGBRegressor,
        config=ModelConfig(params={
            'n_estimators': 200,
            'max_depth': 4,
            'learning_rate': 0.1,
            'n_jobs': 1
        }),
        weight=0.5
    )
    
    enet_reg_spec = ModelSpec(
        model_class=ElasticNet,
        config=ModelConfig(params={
            'alpha': 1.0,
            'l1_ratio': 0.5
        }),
        weight=0.5
    )
    
    ensemble_config = EnsembleConfig(
        default_classifiers=[xgb_clf_spec, lr_clf_spec],
        branch_classifiers=[xgb_clf_spec, lr_clf_spec],
        default_regressors=[xgb_reg_spec, enet_reg_spec],
        branch_regressors=[xgb_reg_spec, enet_reg_spec]
    )


Using Stacking Ensemble...


## 3. Initialize and Run Pipeline

Set up the main configuration object and execute the pipeline.

In [4]:
# 3.1 Main Configuration
config = PredictionConfig(
    market_round=market_round,
    period_type=period_type,
    class_type=class_type,
    
    # Define multiple test periods here
    test_periods=[
        (pd.Timestamp('2023-01'), pd.Timestamp('2023-01')),
        (pd.Timestamp('2023-02'), pd.Timestamp('2023-02')), 
    ],
    
    features=feature_config,
    models=ensemble_config,
    
    training=TrainingConfig(
        min_samples_for_branch_model=10,
        train_months_lookback=12
    ),
    
    paths=DataPathConfig(
        density_path_template=density_path_template,
        constraint_path_template=constraint_path_template
    ),
    
    threshold=ThresholdConfig(threshold_beta=2.0), # Favor Recall
    anomaly_detection=AnomalyDetectionConfig(enabled=True)
)

# 3.2 Initialize Pipeline
pipeline = ShadowPricePipeline(config)

# 3.3 Run Pipeline
results_per_outage, final_results, metrics = pipeline.run(
    verbose=True,
    use_parallel=True,
    n_jobs=2 # Auto-detect
)

SHADOW PRICE PREDICTION PIPELINE - PARALLEL PER-PERIOD TRAINING MODE
Test Periods: 2
  1. Auction: 2023-01, Market: 2023-01
  2. Auction: 2023-02, Market: 2023-02
Class Type: onpeak
Period Type: f0

🚀 Using Ray parallel processing for 2 periods
   Workers: 2

[PARALLEL PROCESSING: Training and Predicting for All Periods]


(PoolActor pid=39378, ip=10.42.7.70) 25-11-19 10:32:20 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=39378, ip=10.42.7.70) 25-11-19 10:32:20 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=39378, ip=10.42.7.70) 25-11-19 10:32:20 EST | INFO    | pbase.config.base    | load_global_config  | #435 | Loading config from local cached json file...
(PoolActor pid=39378, ip=10.42.7.70) 25-11-19 10:32:20 EST | INFO    | pbase.config.base    | __init__            | #44  | Init global config...
(PoolActor pid=39329, ip=10.42.0.182) 25-11-19 10:32:20 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=39329, ip=10.42.0.182) 25-11-19 10:32:20 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=39329, ip=10.42

(PoolActor pid=39378, ip=10.42.7.70) 
(PoolActor pid=39378, ip=10.42.7.70) ================================================================================
(PoolActor pid=39378, ip=10.42.7.70) [Processing Period 2]
(PoolActor pid=39378, ip=10.42.7.70)   Auction Month: 2023-02
(PoolActor pid=39378, ip=10.42.7.70)   Market Month:  2023-02
(PoolActor pid=39378, ip=10.42.7.70) ================================================================================
(PoolActor pid=39378, ip=10.42.7.70) 
(PoolActor pid=39378, ip=10.42.7.70) [STEP 1: Training Period Calculation]
(PoolActor pid=39378, ip=10.42.7.70)   Training Range: 2022-02 to 2023-02
(PoolActor pid=39378, ip=10.42.7.70) 
(PoolActor pid=39378, ip=10.42.7.70) [STEP 2: Loading Training Data]
(PoolActor pid=39378, ip=10.42.7.70) 
(PoolActor pid=39378, ip=10.42.7.70) [Loading Training Data]
(PoolActor pid=39378, ip=10.42.7.70) --------------------------------------------------------------------------------
(PoolActor pid=39378, ip=10.42.7

  0%|          | 0/2 [00:00<?, ?it/s]


[COMBINING RESULTS ACROSS ALL PERIODS]

✓ Results Combined
  Total samples (per-outage): 272,125
  Total unique constraints (monthly): 29,207
  Date range: 2023-01-01 to 2023-02-28

[ANALYZING COMBINED RESULTS]

[MONTHLY AGGREGATED RESULTS]

Test Period: Monthly Level
  Total samples: 29,207

[Shadow Price Prediction Performance]
  MAE:  $1018.86
  RMSE: $4776.14
  MAPE: 16162.47%

[Classification Performance]
  Precision: 0.149
  Recall:    0.198
  F1-Score:  0.170
  Accuracy:  0.839

[Binding Classification Details]
  Correctly classified: 24,505 / 29,207 (83.90%)

  True Positives (correctly identified binding): 481
  False Positives (incorrectly predicted as binding): 2,757
  True Negatives (correctly identified non-binding): 24,024
  False Negatives (missed binding constraints): 1,945

[Binding Rate Analysis]
  Actual binding rate: 8.31% (2,426 samples)
  Predicted binding rate: 11.09% (3,238 samples)

[PER-OUTAGE SUMMARY]

Total outage dates: 21

Average metrics across outage da

## 4. Key Metrics & Analysis

Review the performance of the model.

In [5]:
print("\n" + "="*40)
print("FINAL PERFORMANCE METRICS")
print("="*40)
print(f"Precision: {metrics['monthly']['precision']:.4f}")
print(f"Recall:    {metrics['monthly']['recall']:.4f}")
print(f"F1 Score:  {metrics['monthly']['f1_score']:.4f}")
print(f"MAE:       {metrics['monthly']['mae']:.4f}")
print(f"RMSE:      {metrics['monthly']['rmse']:.4f}")


FINAL PERFORMANCE METRICS
Precision: 0.2671
Recall:    0.4180
F1 Score:  0.3259
MAE:       634.9961
RMSE:      3097.0378


In [5]:
print("\n" + "="*40)
print("FINAL PERFORMANCE METRICS")
print("="*40)
print(f"Precision: {metrics['monthly']['precision']:.4f}")
print(f"Recall:    {metrics['monthly']['recall']:.4f}")
print(f"F1 Score:  {metrics['monthly']['f1_score']:.4f}")
print(f"MAE:       {metrics['monthly']['mae']:.4f}")
print(f"RMSE:      {metrics['monthly']['rmse']:.4f}")


FINAL PERFORMANCE METRICS
Precision: 0.1485
Recall:    0.1983
F1 Score:  0.1698
MAE:       1018.8606
RMSE:      4776.1361
